# Module 3: Discover and combine existing widgets

**Time:** 35 minutes. **Approach: use first, build second.**

Before you build a widget, look for one. In this module you will:

1. see that you already use custom widgets
2. learn a quick way to evaluate any widget
3. work through an example: lasso a cluster, get a DataFrame, link it to controls
4. explore one **station** for your field

## 1. You already use custom widgets (3 minutes)

Plotly's interactive figure is a custom widget. Since Plotly 6 it is built on
anywidget. It can already do lasso selection: choose the lasso tool in the
toolbar, then draw around some points.

In [ ]:
import anywidget
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

rng = np.random.default_rng(0)
points = pd.DataFrame({"x": rng.normal(size=300), "y": rng.normal(size=300)})

fig = go.FigureWidget(go.Scatter(x=points.x, y=points.y, mode="markers"))
fig.update_layout(dragmode="lasso", height=350, margin=dict(l=10, r=10, t=10, b=10))
picked = widgets.Output()


def on_select(trace, selection, state):
    with picked:
        picked.clear_output(wait=True)
        print(f"{len(selection.point_inds)} points selected")
        display(points.iloc[selection.point_inds].describe().loc[["count", "mean"]])


fig.data[0].on_selection(on_select)
print("Is a Plotly FigureWidget an anywidget?", isinstance(fig, anywidget.AnyWidget))
widgets.VBox([fig, picked])

Every widget in today's stations is an anywidget too. One standard, many tools:

In [ ]:
import importlib

examples = {
    "Plotly FigureWidget": ("plotly.graph_objects", "FigureWidget"),
    "jupyter-scatter": ("jscatter.widget", "JupyterScatter"),
    "drawdata": ("drawdata", "ScatterWidget"),
    "quak": ("quak", "Widget"),
    "Altair JupyterChart": ("altair", "JupyterChart"),
    "Lonboard Map": ("lonboard", "Map"),
    "ipyaladin": ("ipyaladin", "Aladin"),
    "vizarr": ("vizarr", "Viewer"),
}
for name, (module, cls) in examples.items():
    try:
        widget_cls = getattr(importlib.import_module(module), cls)
        print(f"{name:22s} anywidget: {issubclass(widget_cls, anywidget.AnyWidget)}")
    except ImportError:
        print(f"{name:22s} (not installed)")

## 2. How to evaluate a widget (5 minutes)

| Question | How to check |
|---|---|
| Does it solve **my** task? | Try its README example with your own data |
| Can I get results **back into Python**? | Look for synced traits or methods (see `widget_report` below) |
| Does it run where I work (JupyterLab, VS Code, Colab, marimo)? | README, or just try it |
| Does it need **internet** in the browser? | Look for `https://` imports in its JavaScript |
| How much data can it handle? | Binary transfer and WebGL scale to millions of points |
| Is it maintained? | Recent release on PyPI, answered issues |
| Licence | Open source and compatible with your work |

`widget_report` answers two of these questions for any widget.

In [ ]:
import re

import altair as alt
import quak
from drawdata import ScatterWidget

IMPORT_URL = re.compile(r"""(?:\bfrom\s*|\bimport\s*\(?\s*)["'](https://[^"']+)["']""")


def widget_report(widget):
    """What can I read back from this widget? Does its JavaScript import code from the internet?"""
    skip = {"layout", "tabbable", "tooltip", "comm", "keys", "log"}
    public = sorted(k for k in widget.keys if not k.startswith("_") and k not in skip)
    remote = sorted(set(IMPORT_URL.findall(str(getattr(widget, "_esm", "")))))
    print(type(widget).__module__ + "." + type(widget).__name__)
    print("  anywidget:           ", isinstance(widget, anywidget.AnyWidget))
    print("  synced public traits:", ", ".join(public) or "(none: uses Python methods instead)")
    print("  imports from the web:", ", ".join(remote) or "none (works offline)")
    print()


for w in [fig, ScatterWidget(), quak.Widget(points), alt.JupyterChart(alt.Chart(points).mark_point().encode(x="x", y="y"))]:
    widget_report(w)

## 3. Worked example: an embedding explorer (15 minutes)

**The task:** you have a 2-D embedding (t-SNE, UMAP, PCA) and want to select a
cluster and get those rows in pandas. We use a widget built for this workshop,
`ScatterExplorer` (in `workshop/scatter_explorer/`).

The data: 1,797 handwritten digits (8×8 images), embedded with t-SNE. This
takes a few seconds.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

from workshop.scatter_explorer import ScatterExplorer

digits = load_digits()
embedding = TSNE(n_components=2, random_state=0).fit_transform(digits.data)

# How many of each point's 10 nearest neighbours share its label?
_, neighbours = NearestNeighbors(n_neighbors=11).fit(embedding).kneighbors(embedding)
agreement = (digits.target[neighbours[:, 1:]] == digits.target[:, None]).mean(axis=1)

df = pd.DataFrame({
    "tsne1": embedding[:, 0],
    "tsne2": embedding[:, 1],
    "digit": digits.target.astype(str),
    "neighbourhood": np.where(agreement < 0.5, "mixed", "clean"),
})
df.head()

**Drag** to lasso a cluster. **Shift + drag** adds. **Wheel** zooms.
**Click a legend entry** to select a whole digit.

In [ ]:
explorer = ScatterExplorer(df, x="tsne1", y="tsne2", color="digit", point_size=4)
explorer

Lasso a cluster, then run this cell:

In [ ]:
explorer.selected_rows["digit"].value_counts()

In [ ]:
def show_digits(rows, max_n=24):
    fig, axes = plt.subplots(3, 8, figsize=(8, 3.3))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, rows.index[:max_n]):
        ax.imshow(digits.images[i], cmap="gray_r")
        ax.set_title(digits.target[i], fontsize=9)
    plt.show()


show_digits(explorer.selected_rows)

### Python to plot

The other direction is just as useful. Select the points whose neighbours
mostly have a different label: likely ambiguous or mislabelled digits.

In [ ]:
explorer.select(df["neighbourhood"] == "mixed")
print(len(explorer.selected), "points sit among other digits")
show_digits(explorer.selected_rows)

### Link the widget to other controls

Now combine it with ipywidgets: a dropdown changes the colouring, and a panel
updates on every selection. This is a second **view** of the same explorer, so
selections in either plot stay in sync.

In [ ]:
color_by = widgets.Dropdown(options=["digit", "neighbourhood"], description="colour by")
panel = widgets.Output()


def on_color(change):
    explorer.set_data(df, "tsne1", "tsne2", color=change["new"])


def on_selection(rows):
    with panel:
        panel.clear_output(wait=True)
        print(f"{len(rows)} selected:", rows["digit"].value_counts().to_dict())
        if len(rows):
            show_digits(rows, max_n=16)


color_by.observe(on_color, names="value")
explorer.on_select(on_selection)
widgets.VBox([color_by, explorer, panel])

### Exercise (5 minutes)

Add a dropdown with the digits `0` to `9` that **selects** all points of the
chosen digit (use `explorer.select(...)`).

In [ ]:
pick_digit = widgets.Dropdown(options=[str(d) for d in range(10)], description="select")

# TODO: when pick_digit changes, select all rows of that digit

pick_digit

## 4. Stations (15 minutes)

Pick **one** station that fits your work. Each notebook is short, shows how to
get results back into Python, and ends with a task that links the widget to
ipywidgets controls.

| Station | Widget | Good for | Internet? |
|---|---|---|---|
| [A](stations/A_jupyter_scatter.ipynb) | jupyter-scatter | Millions of points, embeddings, single-cell data | No |
| [B](stations/B_drawdata.ipynb) | drawdata | Draw a dataset, teach ML intuition | No |
| [C](stations/C_quak.ipynb) | quak | Explore and filter large tables | No |
| [D](stations/D_altair_selections.ipynb) | Altair `JupyterChart` | Chart selections back into pandas | Yes |
| [E](stations/E_lonboard.ipynb) | Lonboard | Geospatial data, maps | Base map only |
| [F](stations/F_ipyaladin.ipynb) | ipyaladin | Astronomy, sky surveys | Yes |
| [G](stations/G_vizarr.ipynb) | vizarr | Large microscopy images (Zarr) | Yes |

More widgets by field: [`resources/widget_catalog.md`](resources/widget_catalog.md).

**Next:** [Module 4, build a custom widget with AI help](04_custom_widget_with_ai.ipynb)